In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [4]:
from scripts.dataset import X_train, X_test, y_train, y_test, groups_train, groups_test, numeric_features, categorical_features

DATASET
Total samples:       6,881
Total trajectories:  750
TRAIN / TEST SPLIT
Train samples:       5,491
Test samples:        1,390
Train trajectories:  600
Test trajectories:   150
TRAIN LABEL DISTRIBUTION
       count  percentage
label                   
-1      1457       26.53
 0       231        4.21
 1      3803       69.26
TEST LABEL DISTRIBUTION
       count  percentage
label                   
-1       402       28.92
 0        56        4.03
 1       932       67.05
LEAKAGE CHECK
Overlapping trajectories: 0
X / y / groups alignment: OK
Trajectory split:          OK


In [5]:
X_train["current_text"]
X_train["context_text"]

X_test["current_text"]
X_test["context_text"]

0                                                        
1       [ASSISTANT]\n<think>We need to determine wheth...
2       [TOOL_RESULT name=search]\n{"result": [[{"docu...
3                                                        
4       [TOOL_CALL]\n\nsearch({"query_list": ["Netherl...
                              ...                        
1385    [TOOL_RESULT name=get_flight_cost]\nError duri...
1386    [TOOL_CALL]\n\nauthenticate_twitter({"username...
1387    [TOOL_CALL]\n\npost_tweet({"content":"Flexibil...
1388    [TOOL_RESULT name=post_tweet]\n{"id": 10, "use...
1389    [TOOL_CALL]\n\nretweet({"tweet_id":10})\n\n[TO...
Name: context_text, Length: 1390, dtype: object

In [6]:
current_train_emb = embedding_model.encode(
    X_train["current_text"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

context_train_emb = embedding_model.encode(
    X_train["context_text"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

current_test_emb = embedding_model.encode(
    X_test["current_text"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

context_test_emb = embedding_model.encode(
    X_test["context_text"].fillna("").tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

Batches:   0%|          | 0/86 [00:00<?, ?it/s]

Batches:   0%|          | 0/86 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [7]:
print(current_train_emb.shape)
print(context_train_emb.shape)

(5491, 384)
(5491, 384)


In [8]:
import numpy as np

X_train_emb = np.hstack([
    current_train_emb,
    context_train_emb,
])

X_test_emb = np.hstack([
    current_test_emb,
    context_test_emb,
])

print(X_train_emb.shape)

(5491, 768)


In [9]:
from sklearn.linear_model import LogisticRegression

embedding_clf = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

embedding_clf.fit(
    X_train_emb,
    y_train,
)

emb_pred = embedding_clf.predict(
    X_test_emb
)

In [10]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print(
    "Accuracy:",
    accuracy_score(y_test, emb_pred)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        emb_pred,
        average="macro",
    )
)

print(
    classification_report(
        y_test,
        emb_pred,
        labels=[-1, 0, 1],
        zero_division=0,
    )
)

print(
    confusion_matrix(
        y_test,
        emb_pred,
        labels=[-1, 0, 1],
    )
)

Accuracy: 0.7683453237410072
Macro F1: 0.5186763920881458
              precision    recall  f1-score   support

          -1       0.69      0.54      0.61       402
           0       1.00      0.05      0.10        56
           1       0.79      0.91      0.85       932

    accuracy                           0.77      1390
   macro avg       0.83      0.50      0.52      1390
weighted avg       0.77      0.77      0.75      1390

[[219   0 183]
 [ 12   3  41]
 [ 86   0 846]]


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_train = scaler.fit_transform(
    X_train[numeric_features]
)

numeric_test = scaler.transform(
    X_test[numeric_features]
)

In [12]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

categorical_train = encoder.fit_transform(
    X_train[categorical_features]
)

categorical_test = encoder.transform(
    X_test[categorical_features]
)

In [13]:
X_train_full_emb = np.hstack([
    current_train_emb,
    context_train_emb,
    numeric_train,
    categorical_train,
])

X_test_full_emb = np.hstack([
    current_test_emb,
    context_test_emb,
    numeric_test,
    categorical_test,
])

In [14]:
embedding_full_clf = LogisticRegression(
    max_iter=5000,
    random_state=42,
)

embedding_full_clf.fit(
    X_train_full_emb,
    y_train,
)

embedding_full_pred = embedding_full_clf.predict(
    X_test_full_emb
)

In [15]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)


def evaluate_model(y_true, y_pred, model_name="Model"):
    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    error_f1 = f1_score(
        y_true,
        y_pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    error_precision = precision_score(
        y_true,
        y_pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    error_recall = recall_score(
        y_true,
        y_pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    print("=" * 60)
    print(model_name)
    print("=" * 60)

    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Macro F1:        {macro_f1:.4f}")
    print(f"Error Precision: {error_precision:.4f}")
    print(f"Error Recall:    {error_recall:.4f}")
    print(f"Error F1:        {error_f1:.4f}")

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[-1, 0, 1],
            zero_division=0,
            digits=4,
        )
    )

    print("Confusion matrix:")
    print(
        confusion_matrix(
            y_true,
            y_pred,
            labels=[-1, 0, 1],
        )
    )

    return {
        "model": model_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "error_precision": error_precision,
        "error_recall": error_recall,
        "error_f1": error_f1,
    }

In [16]:
emb_pred = embedding_clf.predict(
    X_test_emb
)

In [17]:
results_3a = evaluate_model(
    y_test,
    emb_pred,
    model_name="Model 3A — Embeddings only",
)

Model 3A — Embeddings only
Accuracy:        0.7683
Macro F1:        0.5187
Error Precision: 0.6909
Error Recall:    0.5448
Error F1:        0.6092

Classification report:
              precision    recall  f1-score   support

          -1     0.6909    0.5448    0.6092       402
           0     1.0000    0.0536    0.1017        56
           1     0.7907    0.9077    0.8452       932

    accuracy                         0.7683      1390
   macro avg     0.8272    0.5020    0.5187      1390
weighted avg     0.7702    0.7683    0.7470      1390

Confusion matrix:
[[219   0 183]
 [ 12   3  41]
 [ 86   0 846]]


In [18]:
embedding_full_pred = embedding_full_clf.predict(
    X_test_full_emb
)

In [19]:
results_3b = evaluate_model(
    y_test,
    embedding_full_pred,
    model_name="Model 3B — Embeddings + structural features",
)

Model 3B — Embeddings + structural features
Accuracy:        0.8050
Macro F1:        0.5515
Error Precision: 0.7685
Error Recall:    0.6194
Error F1:        0.6860

Classification report:
              precision    recall  f1-score   support

          -1     0.7685    0.6194    0.6860       402
           0     0.6000    0.0536    0.0984        56
           1     0.8172    0.9303    0.8700       932

    accuracy                         0.8050      1390
   macro avg     0.7286    0.5344    0.5515      1390
weighted avg     0.7943    0.8050    0.7857      1390

Confusion matrix:
[[249   1 152]
 [ 11   3  42]
 [ 64   1 867]]


In [20]:
import pandas as pd

embedding_results = pd.DataFrame([
    results_3a,
    results_3b,
])

embedding_results

,model,accuracy,macro_f1,error_precision,error_recall,error_f1
0,Model 3A — Embeddings only,0.768345,0.518676,0.690852,0.544776,0.609179
1,Model 3B — Embeddings + structural features,0.805036,0.551452,0.768519,0.619403,0.685950


In [21]:
def prepare_embedding_dataset(df):
    df = df.copy()

    df["current_text"] = (
        df["current_text"]
        .fillna("")
        .astype(str)
    )

    df["context_text"] = (
        df["context_text"]
        .fillna("")
        .astype(str)
    )

    df["current_role"] = (
        df["current_role"]
        .fillna("UNKNOWN")
        .astype(str)
    )

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce",
    )

    df = (
        df[df["label"].notna()]
        .copy()
        .reset_index(drop=True)
    )

    df["label"] = df["label"].astype(int)

    df["log_context_char_length"] = np.log1p(
        df["context_char_length"]
    )

    df["log_current_char_length"] = np.log1p(
        df["current_char_length"]
    )

    numeric_features = [
        "message_index",
        "previous_messages",
        "previous_tool_calls",
        "previous_tool_results",
        "previous_user_messages",
        "previous_assistant_messages",
        "context_char_length",
        "context_word_count",
        "log_context_char_length",
        "current_char_length",
        "current_word_count",
        "log_current_char_length",
        "is_tool_call",
    ]

    categorical_features = [
        "current_role",
    ]

    return (
        df,
        numeric_features,
        categorical_features,
    )

In [22]:
def run_embedding_cross_dataset(
    train_dfs,
    test_df,
    experiment_name,
):
    # -----------------------------------------
    # Combine training datasets
    # -----------------------------------------

    train_df = pd.concat(
        train_dfs,
        ignore_index=True,
    )

    (
        train_df,
        numeric_features,
        categorical_features,
    ) = prepare_embedding_dataset(
        train_df
    )

    (
        test_df,
        _,
        _,
    ) = prepare_embedding_dataset(
        test_df
    )

    y_train = train_df["label"].to_numpy()
    y_test = test_df["label"].to_numpy()

    # -----------------------------------------
    # 1. Semantic embeddings
    # -----------------------------------------

    print(
        f"\nEncoding {experiment_name}..."
    )

    current_train_emb = embedding_model.encode(
        train_df["current_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    context_train_emb = embedding_model.encode(
        train_df["context_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    current_test_emb = embedding_model.encode(
        test_df["current_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    context_test_emb = embedding_model.encode(
        test_df["context_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    # -----------------------------------------
    # 2. Numeric features
    # Fit ONLY on training data
    # -----------------------------------------

    scaler = StandardScaler()

    numeric_train = scaler.fit_transform(
        train_df[numeric_features]
    )

    numeric_test = scaler.transform(
        test_df[numeric_features]
    )

    # -----------------------------------------
    # 3. Categorical features
    # Fit ONLY on training data
    # -----------------------------------------

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )

    categorical_train = encoder.fit_transform(
        train_df[categorical_features]
    )

    categorical_test = encoder.transform(
        test_df[categorical_features]
    )

    # -----------------------------------------
    # 4. Combine features
    # -----------------------------------------

    X_train_emb = np.hstack([
        current_train_emb,
        context_train_emb,
        numeric_train,
        categorical_train,
    ])

    X_test_emb = np.hstack([
        current_test_emb,
        context_test_emb,
        numeric_test,
        categorical_test,
    ])

    # -----------------------------------------
    # 5. Classifier
    # -----------------------------------------

    classifier = LogisticRegression(
        max_iter=5000,
        random_state=42,
    )

    classifier.fit(
        X_train_emb,
        y_train,
    )

    pred = classifier.predict(
        X_test_emb
    )

    # -----------------------------------------
    # 6. Metrics
    # -----------------------------------------

    accuracy = accuracy_score(
        y_test,
        pred,
    )

    macro_f1 = f1_score(
        y_test,
        pred,
        average="macro",
    )

    error_precision = precision_score(
        y_test,
        pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    error_recall = recall_score(
        y_test,
        pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    error_f1 = f1_score(
        y_test,
        pred,
        labels=[-1],
        average="macro",
        zero_division=0,
    )

    # -----------------------------------------
    # 7. Print
    # -----------------------------------------

    print("\n" + "=" * 80)
    print(experiment_name)
    print("=" * 80)

    print(
        f"Train rows: {len(train_df):,}"
    )

    print(
        f"Test rows: {len(test_df):,}"
    )

    print(
        f"\nAccuracy:        {accuracy:.4f}"
    )

    print(
        f"Macro F1:        {macro_f1:.4f}"
    )

    print(
        f"Error Precision: {error_precision:.4f}"
    )

    print(
        f"Error Recall:    {error_recall:.4f}"
    )

    print(
        f"Error F1:        {error_f1:.4f}"
    )

    print("\nClassification report:")

    print(
        classification_report(
            y_test,
            pred,
            labels=[-1, 0, 1],
            zero_division=0,
            digits=4,
        )
    )

    print("Confusion matrix:")

    print(
        confusion_matrix(
            y_test,
            pred,
            labels=[-1, 0, 1],
        )
    )

    return {
        "experiment": experiment_name,
        "model": "Embeddings + Structural + LogisticRegression",
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "error_precision": error_precision,
        "error_recall": error_recall,
        "error_f1": error_f1,
        "train_rows": len(train_df),
        "test_rows": len(test_df),
    }

In [23]:
from scripts.dataset import context_a, context_b, context_c

embedding_cross_results = []

embedding_cross_results.append(
    run_embedding_cross_dataset(
        train_dfs=[
            context_a,
            context_b,
        ],
        test_df=context_c,
        experiment_name="A+B -> C",
    )
)

embedding_cross_results.append(
    run_embedding_cross_dataset(
        train_dfs=[
            context_a,
            context_c,
        ],
        test_df=context_b,
        experiment_name="A+C -> B",
    )
)

embedding_cross_results.append(
    run_embedding_cross_dataset(
        train_dfs=[
            context_b,
            context_c,
        ],
        test_df=context_a,
        experiment_name="B+C -> A",
    )
)


Encoding A+B -> C...


Batches:   0%|          | 0/68 [00:00<?, ?it/s]

Batches:   0%|          | 0/68 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/97 [00:00<?, ?it/s]

Batches:   0%|          | 0/97 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]


B+C -> A
Train rows: 6,147
Test rows: 734

Accuracy:        0.6689
Macro F1:        0.3901
Error Precision: 0.4420
Error Recall:    0.3408
Error F1:        0.3849

Classification report:
              precision    recall  f1-score   support

          -1     0.4420    0.3408    0.3849       179
           0     0.0000    0.0000    0.0000        56
           1     0.7215    0.8617    0.7854       499

    accuracy                         0.6689       734
   macro avg     0.3878    0.4008    0.3901       734
weighted avg     0.5983    0.6689    0.6278       734

Confusion matrix:
[[ 61   0 118]
 [  8   0  48]
 [ 69   0 430]]


In [24]:
embedding_cross_df = pd.DataFrame(
    embedding_cross_results
)

embedding_cross_df[
    [
        "experiment",
        "accuracy",
        "macro_f1",
        "error_precision",
        "error_recall",
        "error_f1",
        "train_rows",
        "test_rows",
    ]
]

,experiment,accuracy,macro_f1,error_precision,error_recall,error_f1,train_rows,test_rows
0,A+B -> C,0.522780,0.314610,0.216771,0.426316,0.287404,4291,2590
1,A+C -> B,0.654765,0.398765,0.491564,0.393694,0.437219,3324,3557
2,B+C -> A,0.668937,0.390082,0.442029,0.340782,0.384858,6147,734


This answers the Model 3 hypothesis pretty clearly: **frozen MiniLM embeddings did not solve cross-dataset generalization.**

| Experiment | TF-IDF + LinearSVC | Embeddings + structural | Change |
| ---------- | -----------------: | ----------------------: | -----: |
| A+B → C    |              0.300 |               **0.315** | +0.015 |
| A+C → B    |              0.382 |               **0.399** | +0.017 |
| B+C → A    |          **0.420** |                   0.390 | −0.030 |

So embeddings give tiny improvements on two held-out datasets and regress on the third. That's not convincing evidence of better generalization.

More importantly, your regular held-out result also dropped substantially:

```text
TF-IDF + LinearSVC
Macro F1 = 0.655

Frozen embeddings + structural
Macro F1 = 0.551
```

### What you've learned

Your experiments are now isolating the problem nicely:

```text
Model 1
TF-IDF + Logistic
       ↓
Model 2
TF-IDF + LinearSVC
       ↓
In-domain improves
Cross-domain remains poor

Model 3
Frozen semantic embeddings
       ↓
In-domain gets worse
Cross-domain remains poor
```

So the evidence is pointing toward something deeper than simply "TF-IDF doesn't understand synonyms."

A likely issue is that detecting agent failures requires understanding the **relationship between context and current action**.

For example:

```text
CONTEXT
[TOOL_RESULT]
payment_status = rejected

CURRENT
[ASSISTANT]
Your payment was successful.
```

Your current embedding approach independently does:

```text
embedding("payment_status = rejected")
                 +
embedding("Your payment was successful")
                 ↓
          LogisticRegression
```

The embedding model never jointly reads the contradiction. You're hoping the downstream linear classifier can reconstruct that relationship from two independent vectors.

That's a significant limitation.